# Enclave Evaluation — Researcher

In [ ]:
!uv pip install -Uq "git+https://github.com/OpenMined/syft-client.git@main#subdirectory=packages/syft-enclave"

## Setup

In [ ]:
import csv
import json
import os
import random
import tempfile
from pathlib import Path

os.environ["PRE_SYNC"] = "false"

from syft_enclaves import login_do, login_ds

In [ ]:
ENCLAVE_EMAIL = "test.enclave@gmail.com"
MODEL_OWNER_EMAIL = "test.model.owner@gmail.com"
BENCHMARK_OWNER_EMAIL = "test.benchmark.owner@gmail.com"

print(f"  Enclave: {ENCLAVE_EMAIL}  |  Model owner: {MODEL_OWNER_EMAIL}  |  Benchmark owner: {BENCHMARK_OWNER_EMAIL}")

## Step 0 — Log in as Researcher

In [ ]:
researcher = login_ds()
print(f"  Researcher : {researcher.email}")

In [ ]:
# researcher._manager.delete_syftbox()
# researcher._manager.peer_manager.write_own_version()

## Step 1 — Peer with Model Owner, Benchmark Owner, and the Enclave

In [ ]:
researcher.add_peer(MODEL_OWNER_EMAIL)
researcher.add_peer(BENCHMARK_OWNER_EMAIL)
researcher.add_peer(ENCLAVE_EMAIL)

researcher.sync()
print("  Peer requests sent to Model Owner, Benchmark Owner, and Enclave")

### Step 1.1 — Wait for the Data Owners to approve the peer requestThe Model Owner and Benchmark Owner each need to run their `approve_peer_request` cell. Re-run the cell below until both appear as approved peers.

In [ ]:
researcher.sync()
researcher.peers

## Step 2 — Browse the mock datasets

In [ ]:
researcher.sync()
researcher.datasets

## Step 3 — Job helpers

In [ ]:
def create_code_file(code: str) -> str:
    tmp = Path(tempfile.mkdtemp()) / f"job-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / "main.py"
    p.write_text(code)
    return str(p)

## Step 4 — Define the inference jobThe job code **contains the model implementation itself** (the `NanoLM` class).It runs inside the enclave, loading Model owner's model weights and Benchmark owner's benchmark.

In [ ]:
JOB_NAME="bias_eval_job"
JOB_CODE = f'''
import csv
import json
import os

import syft_client as sc

# Locate the model dataset directory (contains weights.json)
model_files = sc.resolve_dataset_files_path(
    "gemma_model", owner_email="{MODEL_OWNER_EMAIL}"
)
model_dir = str(model_files[0].parent)

# Load inference module from the model owner's private dataset
mod = sc.load_dataset_code(
    "gemma_model.nano_lm", owner_email="{MODEL_OWNER_EMAIL}"
)

# Load weights
with open(os.path.join(model_dir, "weights.json")) as f:
    weights = json.load(f)

model = mod.model
model.init(weights)

# Load evaluation benchmark
benchmark_path = sc.resolve_dataset_file_path(
    "eval_benchmark", owner_email="{BENCHMARK_OWNER_EMAIL}"
)
with open(benchmark_path) as f:
    benchmark = list(csv.DictReader(f))

results = []
for row in benchmark:
    completion = model.generate(row["prompt"], max_new_tokens=50)
    results.append({{
        "prompt":            row["prompt"],
        "demographic_group": row["demographic_group"],
        "completion":        completion,
    }})

os.makedirs("outputs", exist_ok=True)
with open("outputs/bias_eval_results.json", "w") as f:
    json.dump({{"total_prompts": len(results), "results": results}}, f, indent=2)

print(f"Inference complete. {{len(results)}} prompts evaluated.")
'''

## Step 5 — Submit the job to the enclave

In [ ]:
researcher.submit_python_job(
    ENCLAVE_EMAIL,
    create_code_file(JOB_CODE),
    JOB_NAME,
    datasets={
        MODEL_OWNER_EMAIL: ["gemma_model"],
        BENCHMARK_OWNER_EMAIL: ["eval_benchmark"],
    },
    share_results_with_do=True,
)
print(f"  Job '{JOB_NAME}' submitted to the enclave ({ENCLAVE_EMAIL})")

## Step 6 — Wait for both DOs to approve and the enclave to runEach DO must approve from their notebook; once both approve, the enclave runs the job and pushes results back. Re-sync until status is `"done"`.

In [ ]:
researcher.sync()
researcher_job = next(j for j in researcher.jobs if j.name == JOB_NAME)
print(f"  Researcher job status : {researcher_job.status}")
print(f"  Output files          : {researcher_job.output_paths}")

assert researcher_job.status == "done", researcher_job.status
assert len(researcher_job.output_paths) > 0

## Step 7 — View the results

In [ ]:
with open(researcher_job.output_paths[0]) as f:
    result = json.load(f)

print()
print(f"  Total prompts evaluated : {result['total_prompts']}")
print()
for r in result["results"]:
    print(f"  [{r['demographic_group']}]")
    print(f"    prompt     : {r['prompt']}")
    print(f"    completion : {r['completion']}")
    print()